# SMS Spam Detection with DistilBERT

This beginner-friendly notebook uses **transfer learning** to fine-tune a pretrained DistilBERT model on the UCI SMS Spam Collection dataset. It evaluates the model, lets you test custom messages, and exports the model for the FastAPI web app.

> In Colab, choose **Runtime → Change runtime type → T4 GPU** before starting.

## Step 1 — Install the libraries

Transformers supplies DistilBERT, Datasets prepares the text, and scikit-learn calculates our evaluation metrics.

In [ ]:
%pip install -q transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 scikit-learn==1.5.2 ucimlrepo==0.0.7

## Step 2 — Import the tools and set a random seed

Using one seed makes the train/test split repeatable.

In [ ]:
import json
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "distilbert/distilbert-base-uncased"
set_seed(SEED)
print("GPU available:", torch.cuda.is_available())

## Step 3 — Download and inspect the SMS Spam Collection

The dataset contains genuine messages labeled `ham` and unwanted messages labeled `spam`. We map ham to 0 and spam to 1.

In [ ]:
sms_data = fetch_ucirepo(id=228)

# The UCI package returns the messages and labels as separate DataFrames.
messages = sms_data.data.features.iloc[:, 0].astype(str)
label_names = sms_data.data.targets.iloc[:, 0].astype(str).str.lower()

data = pd.DataFrame({"text": messages, "label_name": label_names})
data["label"] = data["label_name"].map({"ham": 0, "spam": 1})
data = data.dropna(subset=["label"])[["text", "label"]]
data["label"] = data["label"].astype(int)

print("Number of messages:", len(data))
print(data["label"].value_counts().rename(index={0: "Ham", 1: "Spam"}))
data.sample(5, random_state=SEED)

## Step 4 — Split into training and testing sets

We use 80% for training and 20% for testing. `stratify` keeps a similar spam/ham balance in both sets.

In [ ]:
train_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"],
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Training messages: {len(train_dataset):,}")
print(f"Testing messages:  {len(test_dataset):,}")

## Step 5 — Tokenize the messages

A tokenizer turns words into the numeric IDs DistilBERT understands. SMS messages are short, so a maximum length of 128 tokens is enough for this project.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Step 6 — Load DistilBERT and define the metrics

The pretrained model already understands language. We add a two-class classification head and fine-tune the whole model for ham versus spam. Precision, recall, and F1 are calculated for the spam class.

In [ ]:
id2label = {0: "HAM", 1: "SPAM"}
label2id = {"HAM": 0, "SPAM": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(evaluation):
    logits, true_labels = evaluation
    predicted_labels = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## Step 7 — Fine-tune the model

Three epochs are a good simple starting point. The trainer evaluates after every epoch and keeps the checkpoint with the best F1-score. This usually takes a few minutes on a Colab T4 GPU.

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-spam-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 8 — Evaluate on the test set

Accuracy measures all correct predictions. Precision asks how often predicted spam was really spam. Recall asks how much real spam we found. F1 balances precision and recall.

In [ ]:
test_results = trainer.evaluate(tokenized_test)
print("Test metrics")
for metric in ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]:
    print(f"{metric.replace('eval_', '').title():10s}: {test_results[metric]:.4f}")

prediction_output = trainer.predict(tokenized_test)
predicted_labels = np.argmax(prediction_output.predictions, axis=-1)
true_labels = test_df["label"].to_numpy()

print("\nDetailed classification report:")
print(classification_report(true_labels, predicted_labels, target_names=["Ham", "Spam"], digits=4))
print("Confusion matrix [[ham correct, ham as spam], [spam as ham, spam correct]]:")
print(confusion_matrix(true_labels, predicted_labels))

## Step 9 — Try your own message

Change the text below and run the cell again.

In [ ]:
def predict_message(message):
    model.eval()
    inputs = tokenizer(
        message, return_tensors="pt", truncation=True, max_length=128
    ).to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    predicted_id = int(np.argmax(probabilities))
    label = "Spam" if predicted_id == 1 else "Not Spam (Ham)"
    print(f"Prediction: {label}")
    print(f"Confidence: {probabilities[predicted_id]:.2%}")
    print(f"Ham probability: {probabilities[0]:.2%}")
    print(f"Spam probability: {probabilities[1]:.2%}")
    return label

custom_message = "Congratulations! You won a free ticket. Call now to claim."
predict_message(custom_message)

## Step 10 — Save and download the model for the web app

This creates a ZIP containing the model, tokenizer, and test metrics. Download it, then extract its contents into the project's `model/` folder.

In [ ]:
EXPORT_DIR = "spam-distilbert-model"
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

metrics_to_save = {
    "accuracy": round(test_results["eval_accuracy"], 6),
    "precision": round(test_results["eval_precision"], 6),
    "recall": round(test_results["eval_recall"], 6),
    "f1": round(test_results["eval_f1"], 6),
}
with open(f"{EXPORT_DIR}/metrics.json", "w") as file:
    json.dump(metrics_to_save, file, indent=2)

shutil.make_archive(EXPORT_DIR, "zip", EXPORT_DIR)
print(f"Created {EXPORT_DIR}.zip")

from google.colab import files
files.download(f"{EXPORT_DIR}.zip")